In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("DockerJupyterSpark") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/12 04:37:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.csv("/opt/spark-data/tips.csv", header=True, inferSchema=True)

In [4]:
df.show()

+----------+----+------+------+---+------+----+
|total_bill| tip|   sex|smoker|day|  time|size|
+----------+----+------+------+---+------+----+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|
|     26.88|3.12|Female|    No|Sun|Dinner|   4|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|
|     14.78|3.23|  Male|    No|Sun|Dinner|   2|
+----------+----+------+------+---+------+----+



In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HiveTest")
    .master("spark://spark-master:7077")

    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")

    .config("spark.driver.port", "37771")
    .config("spark.blockManager.port", "37772")

    .config("spark.sql.warehouse.dir", "/opt/spark-notebooks/warehouse")
    .enableHiveSupport()
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/20 18:19:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
spark.sparkContext.getConf().get("spark.driver.port")

'37771'

In [3]:
spark.range(10).count()

10

In [4]:
spark.sql("SHOW DATABASES").show()

26/06/20 18:26:21 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/06/20 18:26:21 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/06/20 18:26:24 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/06/20 18:26:24 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@172.21.0.8
26/06/20 18:26:24 WARN ObjectStore: Failed to get database default, returning NoSuchObjectException


+---------+
|namespace|
+---------+
|  default|
+---------+



In [5]:
spark.sql("CREATE DATABASE IF NOT EXISTS test_db")

26/06/20 18:26:45 WARN ObjectStore: Failed to get database test_db, returning NoSuchObjectException
26/06/20 18:26:45 WARN ObjectStore: Failed to get database test_db, returning NoSuchObjectException
26/06/20 18:26:45 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
26/06/20 18:26:45 WARN ObjectStore: Failed to get database test_db, returning NoSuchObjectException
chgrp: changing ownership of 'file:///opt/spark-notebooks/warehouse': chown: changing group of '/opt/spark-notebooks/warehouse': Operation not permitted


DataFrame[]

In [6]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
|  test_db|
+---------+



In [8]:
spark.sql("SHOW TABLES").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  test_db| students|      false|
+---------+---------+-----------+



In [9]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
|  test_db|
+---------+



In [10]:
spark.sql("""
CREATE TABLE IF NOT EXISTS test_db.students (
    id INT,
    name STRING
)
""")

26/06/20 18:30:10 WARN ResolveSessionCatalog: A Hive serde table will be created as there is no table provider specified. You can set spark.sql.legacy.createHiveTableByDefault to false so that native data source table will be created instead.


DataFrame[]

In [11]:
spark.sql("SHOW TABLES IN test_db").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  test_db| students|      false|
+---------+---------+-----------+



In [13]:
spark.sql("SELECT * FROM test_db.students").show()

+---+----+
| id|name|
+---+----+
+---+----+



In [14]:
spark.conf.get("spark.sql.warehouse.dir")

'file:/opt/spark-notebooks/warehouse'

In [15]:
spark.sql("DESCRIBE FORMATTED test_db.students").show(200, False)

+----------------------------+----------------------------------------------------------+-------+
|col_name                    |data_type                                                 |comment|
+----------------------------+----------------------------------------------------------+-------+
|id                          |int                                                       |NULL   |
|name                        |string                                                    |NULL   |
|                            |                                                          |       |
|# Detailed Table Information|                                                          |       |
|Catalog                     |spark_catalog                                             |       |
|Database                    |test_db                                                   |       |
|Table                       |students                                                  |       |
|Owner              

In [16]:
spark.sql("DROP TABLE test_db.students")

DataFrame[]

In [19]:
spark.sql("SHOW DATABASES").show(truncate=False)

+---------+
|namespace|
+---------+
|default  |
|test_db  |
+---------+



In [20]:
spark.sql("DESCRIBE DATABASE EXTENDED test_db").show(truncate=False)

+--------------+----------------------------------------------+
|info_name     |info_value                                    |
+--------------+----------------------------------------------+
|Catalog Name  |spark_catalog                                 |
|Namespace Name|test_db                                       |
|Comment       |                                              |
|Location      |file:/opt/spark-notebooks/warehouse/test_db.db|
|Owner         |hdfs                                          |
|Properties    |                                              |
+--------------+----------------------------------------------+



In [21]:
spark.sql("DROP TABLE IF EXISTS test_db.students")

DataFrame[]

In [22]:
spark.sql("DROP TABLE IF EXISTS test_db.students")

DataFrame[]

In [23]:
spark.sql("DROP TABLE IF EXISTS test_db.students")

DataFrame[]

In [24]:
spark.sql("DROP DATABASE IF EXISTS test_db")

26/06/20 18:54:23 WARN TxnHandler: Cannot perform cleanup since metastore table does not exist


DataFrame[]

In [25]:
spark.sql("""
CREATE DATABASE test_db
LOCATION 'hdfs://hadoop-master:9000/user/hive/warehouse/test_db.db'
""")

26/06/20 18:55:15 WARN ObjectStore: Failed to get database test_db, returning NoSuchObjectException
26/06/20 18:55:15 WARN ObjectStore: Failed to get database test_db, returning NoSuchObjectException
26/06/20 18:55:15 WARN ObjectStore: Failed to get database test_db, returning NoSuchObjectException


DataFrame[]

In [26]:
spark.sql("DESCRIBE DATABASE EXTENDED test_db").show(truncate=False)

+--------------+--------------------------------------------------------+
|info_name     |info_value                                              |
+--------------+--------------------------------------------------------+
|Catalog Name  |spark_catalog                                           |
|Namespace Name|test_db                                                 |
|Comment       |                                                        |
|Location      |hdfs://hadoop-master:9000/user/hive/warehouse/test_db.db|
|Owner         |hdfs                                                    |
|Properties    |                                                        |
+--------------+--------------------------------------------------------+



In [27]:
spark.sql("""
CREATE TABLE test_db.students (
    id INT,
    name STRING
)
USING hive
""")

26/06/20 18:58:14 WARN HiveMetaStore: Location: hdfs://hadoop-master:9000/user/hive/warehouse/test_db.db/students specified for non-external table:students


DataFrame[]

In [28]:
spark.sql("SHOW TABLES IN test_db").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  test_db| students|      false|
+---------+---------+-----------+



In [29]:
spark.sql("""
INSERT INTO test_db.students
VALUES
(1,'Ahmed'),
(2,'Ali')
""")

DataFrame[]

In [30]:
spark.sql("SELECT * FROM test_db.students").show()

+---+-----+
| id| name|
+---+-----+
|  1|Ahmed|
|  2|  Ali|
+---+-----+



In [31]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
|  test_db|
+---------+



In [32]:
spark.sql("SHOW TABLES IN test_db").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  test_db| students|      false|
+---------+---------+-----------+



In [33]:
spark.sql("DESCRIBE FORMATTED test_db.students").show(100,False)

+----------------------------+-----------------------------------------------------------------+-------+
|col_name                    |data_type                                                        |comment|
+----------------------------+-----------------------------------------------------------------+-------+
|id                          |int                                                              |NULL   |
|name                        |string                                                           |NULL   |
|                            |                                                                 |       |
|# Detailed Table Information|                                                                 |       |
|Catalog                     |spark_catalog                                                    |       |
|Database                    |test_db                                                          |       |
|Table                       |students                 

In [34]:
spark.version

'3.5.1'

In [35]:
spark.sparkContext.getConf().getAll()

[('spark.driver.extraJavaOptions',
  '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false'),
 ('spark.blockManager.port', '37772'),
 ('spark.driver.host', 'jupyter'),
 ('spark.app.name', 'HiveTest

In [38]:
spark.stop()
